In [21]:
!pip -q install streamlit
!pip -q install openai-whisper
!pip -q install sentence-transformers
!pip -q install reportlab
!pip -q install pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 90.3 MB/s eta 0:00:00


In [22]:
%%writefile app.py

import streamlit as st
import whisper
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from reportlab.pdfgen import canvas
import tempfile

st.set_page_config(
    page_title="Voice Based Concept Understanding Analyser",
    page_icon="🎤",
    layout="wide"
)

st.title("🎤 Voice Based Concept Understanding Analyser")

st.markdown("---")

st.write("Analyze Concept Understanding using Artificial Intelligence")

@st.cache_resource
def load_models():
    whisper_model = whisper.load_model("base")
    embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
    return whisper_model, embedding_model

whisper_model, embedding_model = load_models()

reference = st.text_area(
    "Enter Reference Answer",
    height=180
)

audio = st.file_uploader(
    "Upload Voice Answer",
    type=["wav","mp3","m4a"]
)

if st.button("Analyze"):

    if reference == "":
        st.warning("Please enter the reference answer.")
        st.stop()

    if audio is None:
        st.warning("Please upload an audio file.")
        st.stop()

    with tempfile.NamedTemporaryFile(delete=False) as tmp:
        tmp.write(audio.read())
        audio_path = tmp.name

    with st.spinner("Converting Speech to Text..."):
        result = whisper_model.transcribe(audio_path)

    spoken_text = result["text"]

    st.subheader("Recognized Speech")

    st.success(spoken_text)

    ref_embedding = embedding_model.encode([reference])

    user_embedding = embedding_model.encode([spoken_text])

    similarity = cosine_similarity(
        ref_embedding,
        user_embedding
    )[0][0]

    score = similarity * 100

    st.subheader("Concept Understanding Score")

    st.progress(int(score))

    st.metric(
        "Similarity",
        f"{score:.2f}%"
    )

    if score >= 85:
        feedback = "Excellent Understanding ✅"

    elif score >= 70:
        feedback = "Good Understanding 👍"

    elif score >= 50:
        feedback = "Average Understanding ⚠"

    else:
        feedback = "Needs Improvement ❌"

    st.success(feedback)

    pdf = canvas.Canvas("Report.pdf")

    pdf.drawString(
        80,
        800,
        "VOICE BASED CONCEPT UNDERSTANDING ANALYSER"
    )

    pdf.drawString(
        80,
        760,
        f"Similarity Score : {score:.2f}%"
    )

    pdf.drawString(
        80,
        730,
        "Feedback :"
    )

    pdf.drawString(
        80,
        710,
        feedback
    )

    pdf.save()

    with open("Report.pdf","rb") as f:
        st.download_button(
            "📄 Download Report",
            f,
            file_name="Concept_Report.pdf"
        )

Writing app.py


In [23]:
from pyngrok import ngrok
import threading
import os

def run():
    os.system("streamlit run app.py --server.port 8501")

thread = threading.Thread(target=run)
thread.start()

In [26]:
from pyngrok import ngrok

# Replace 'YOUR_AUTHTOKEN' with your actual ngrok authentication token
ngrok.set_auth_token('3GUiBsciUqUbI1oBzAf0NqhM4l4_53S6oSjESqzjdAFgYu65L')

public_url = ngrok.connect(8501)

print(public_url)

NgrokTunnel: "https://mystified-twerp-evacuate.ngrok-free.dev" -> "http://localhost:8501"


In [11]:
from google.colab import files

uploaded = files.upload()

Saving song.mpeg to song.mpeg


In [12]:
audio_file = list(uploaded.keys())[0]

result = whisper_model.transcribe(audio_file)

spoken_text = result["text"]

print("Recognized Speech:")
print(spoken_text)

/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Recognized Speech:
 When all I dream of is yours All I long for is your touch And all in something tells me it's love If you can say that I'm a fool And I don't know very much But I think they call this love


In [13]:
reference_text = """
Machine Learning is a branch of Artificial Intelligence that allows computers to learn from data without being explicitly programmed.
"""

In [17]:
reference_embedding = model.encode([reference_text])

spoken_embedding = model.encode([spoken_text])

similarity = cosine_similarity(reference_embedding, spoken_embedding)[0][0]

score = similarity * 100

print("Concept Understanding Score:", round(score,2),"%")

Concept Understanding Score: 5.21 %


In [19]:
if score >= 85:
    feedback = "Excellent Understanding"
elif score >= 70:
    feedback = "Good Understanding"
elif score >= 50:
    feedback = "Average Understanding"
else:
    feedback = "Needs Improvement"

print(feedback)

Needs Improvement


In [20]:
from reportlab.pdfgen import canvas

pdf = canvas.Canvas("Concept_Report.pdf")

pdf.drawString(80,800,"VOICE BASED CONCEPT UNDERSTANDING ANALYSER")

pdf.drawString(80,760,f"Score : {score:.2f}%")

pdf.drawString(80,730,"Feedback:")

pdf.drawString(80,710,feedback)

pdf.save()

print("PDF Generated Successfully")

PDF Generated Successfully
